In [7]:
pip install mlflow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### State a Tracking Server

In [3]:
!mlflow server --host 127.0.0.1 --port 8080

^C


# Set the Tracking Server URI

In [4]:
import mlflow

mlflow.set_tracking_uri(uri = 'http://0.0.0.0:85')

# Step 3 - Train a model and prepare metadata for logging

### Load and prepare the Iris dataset for modeling.
### Train a Logistic Regression model and evaluate its performance.
### Prepare the model hyperparameters and calculate metrics for logging.

In [13]:
import mlflow
from mlflow.models import infer_signature

import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

X, y = datasets.load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state = 42
)

params = {
    "solver": "lbfgs",
    "max_iter": 1000,
    "multi_class": "auto",
    "random_state": 8888,
}

lr = LogisticRegression(**params)
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

C:\Users\ravin\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


### Step 4 - Log the model and its metadata to MLflow

In [18]:
mlflow.set_tracking_uri(uri = 'http://127.0.0.1:8080')

# Creation of new MLflow experiment
mlflow.set_experiment("ML flow Quickstart")

# Starting mlflow run
with mlflow.start_run():
    mlflow.log_params(params)
    mlflow.log_metric("accuracy", accuracy)

    mlflow.set_tag("Training Info", "Basic LR model for iris data")

    signature = infer_signature(X_train, lr.predict(X_train))

    model_info = mlflow.sklearn.log_model(
        sk_model = lr,
        artifact_path = "iris_model",
        signature = signature,
        input_example = X_train,
        registered_model_name = "tracking-quickstart",
    )


2025/05/03 11:00:54 INFO mlflow.tracking.fluent: Experiment with name 'ML flow Quickstart' does not exist. Creating a new experiment.


Successfully registered model 'tracking-quickstart'.
2025/05/03 11:01:03 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: tracking-quickstart, version 1


🏃 View run loud-croc-610 at: http://127.0.0.1:8080/#/experiments/373220069622949787/runs/021c5f733e8d488995390c305a436e1f
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/373220069622949787


Created version '1' of model 'tracking-quickstart'.


### Step 5 - Load the model as a Python Function (pyfunc) and use it for inference

In [29]:
# Load the model back for predictions as a generic Python Function model

loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)
predictions = loaded_model.predict(X_test)

iris_feature_names = datasets.load_iris().feature_names

result = pd.DataFrame(X_test, columns=iris_feature_names)
result['actual_class'] = y_test
result['prediction_class'] = predictions
result[:4]

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),actual_class,prediction_class
0,6.1,2.8,4.7,1.2,1,1
1,5.7,3.8,1.7,0.3,0,0
2,7.7,2.6,6.9,2.3,2,2
3,6.0,2.9,4.5,1.5,1,1
